In [3]:
import pandas as pd
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "FIFA WC Dataset"
CLEANED_DATA = PROJECT_ROOT / "cleaned_data"
PROCESSED_DATA = PROJECT_ROOT / "processed_data"

results = pd.read_csv(CLEANED_DATA / "results_2000_onwards.csv")
goalscorer = pd.read_csv(CLEANED_DATA / "goalscorers_2000_onwards.csv")
shootouts = pd.read_csv(CLEANED_DATA / "shootouts_2000_onwards.csv")

In [4]:
print(results.count())

date          25386
home_team     25386
away_team     25386
home_score    25386
away_score    25386
tournament    25386
city          25386
country       25386
neutral       25386
dtype: int64


In [5]:
results_played = results.dropna(subset = ["home_score", "away_score"]).copy()
print(results_played.count())

date          25386
home_team     25386
away_team     25386
home_score    25386
away_score    25386
tournament    25386
city          25386
country       25386
neutral       25386
dtype: int64


In [6]:
print(results_played.shape)

(25386, 9)


In [7]:
home_df = results_played[
    ["date", "home_team", "away_team", "home_score", "away_score", "tournament", "neutral"
     ]].copy()

home_df = home_df.rename(columns = {"home_team": "team", "away_team":"opponent", "home_score": "goals_for", "away_score": "goals_against"}) 
home_df.head()

,date,team,opponent,goals_for,goals_against,tournament,neutral
0,04-01-2000,Egypt,Togo,2,1,Friendly,False
1,07-01-2000,Tunisia,Togo,7,0,Friendly,False
2,08-01-2000,Trinidad and Tobago,Canada,0,0,Friendly,False
3,09-01-2000,Burkina Faso,Gabon,1,1,Friendly,False
4,09-01-2000,Guatemala,Armenia,1,1,Friendly,True


In [8]:
away_df = results_played[[
    "date", "home_team", "away_team", "home_score", "away_score", "tournament", "neutral"
]].copy()
away_df = away_df.rename(columns = {"home_team": "opponent", "away_team": "team", "home_score": "goals_against", "away_score": "goals_for"})

away_df.head()

,date,opponent,team,goals_against,goals_for,tournament,neutral
0,04-01-2000,Egypt,Togo,2,1,Friendly,False
1,07-01-2000,Tunisia,Togo,7,0,Friendly,False
2,08-01-2000,Trinidad and Tobago,Canada,0,0,Friendly,False
3,09-01-2000,Burkina Faso,Gabon,1,1,Friendly,False
4,09-01-2000,Guatemala,Armenia,1,1,Friendly,True


In [9]:
team_matches = pd.concat([
    home_df, away_df
], ignore_index = True)

team_matches[
    ((team_matches["team"] == "Egypt") &
     (team_matches["opponent"] == "Togo")) |
    ((team_matches["team"] == "Togo") &
     (team_matches["opponent"] == "Egypt"))
]

,date,team,opponent,goals_for,goals_against,tournament,neutral
0,04-01-2000,Egypt,Togo,2,1,Friendly,False
16,19-01-2000,Togo,Egypt,1,0,Friendly,False
11704,22-05-2012,Egypt,Togo,3,0,Friendly,True
16477,28-03-2017,Egypt,Togo,3,0,Friendly,False
19572,14-11-2020,Egypt,Togo,1,0,African Cup of Nations qualification,False
19634,17-11-2020,Togo,Egypt,1,3,African Cup of Nations qualification,False
25386,04-01-2000,Togo,Egypt,1,2,Friendly,False
25402,19-01-2000,Egypt,Togo,0,1,Friendly,False
37090,22-05-2012,Togo,Egypt,0,3,Friendly,True
41863,28-03-2017,Togo,Egypt,0,3,Friendly,False


In [10]:
def get_result(row):
    if row["goals_for"] > row["goals_against"]:
        return "W"
    elif row["goals_for"] < row["goals_against"]:
        return "L"
    else:
        return "D"

team_matches["result"] = team_matches.apply(
    get_result,
    axis=1
)

#print(team_matches.iloc[0])
#print(team_matches.iloc[25313])
print(team_matches["result"].value_counts())

result
W    19468
L    19468
D    11836
Name: count, dtype: int64


In [11]:
team_matches.info()
team_matches.head()
team_matches.shape

<class 'pandas.DataFrame'>
RangeIndex: 50772 entries, 0 to 50771
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   date           50772 non-null  str  
 1   team           50772 non-null  str  
 2   opponent       50772 non-null  str  
 3   goals_for      50772 non-null  int64
 4   goals_against  50772 non-null  int64
 5   tournament     50772 non-null  str  
 6   neutral        50772 non-null  bool 
 7   result         50772 non-null  str  
dtypes: bool(1), int64(2), str(5)
memory usage: 2.8 MB


(50772, 8)

In [12]:
team_matches.to_csv(PROCESSED_DATA / "team_matches.csv", index = False)


In [13]:
print(team_matches[(team_matches["team"] == "South Africa") & (team_matches["opponent"] == "Canada")])

             date          team opponent  goals_for  goals_against  \
7416   20-11-2007  South Africa   Canada          2              0   
25382  28-06-2026  South Africa   Canada          0              1   

           tournament  neutral result  
7416         Friendly    False      W  
25382  FIFA World Cup     True      L  


In [14]:
team_matches.shape

(50772, 8)

In [15]:
team_matches["date"] = pd.to_datetime(team_matches["date"], dayfirst = True)
team_matches.to_csv(PROCESSED_DATA / "team_matches.csv", index = False)

## Session Summary

Created a team-centric dataset (`team_matches.csv`) from `results_2000_onwards.csv` by transforming each match into two team perspectives (home and away). Added `goals_for`, `goals_against`, and `result` (W/D/L) columns.

Included `date`, `tournament`, and `neutral` fields for future chronological analysis and Elo calculations. Final dataset contains **50,678 rows** representing **25,339 played matches**.

Saved the output to `processed_data/team_matches.csv`. Next step: create `team_stats.csv` and implement Elo ratings.

Matche will be updated daily after each FIFA WC 26 Match.